# Streaming Reservoir Computing From TDMS States

This notebook trains the readout directly from measured TDMS reservoir states.
The modification follows the `RC_MGV1.ipynb` readout idea: use a fixed sampled
camera/state feature vector and train the output by ridge regression after a
washout period. The TDMS frames are still streamed, so the full state matrix is
not kept in memory.


In [ ]:
from pathlib import Path
import glob
import numpy as np
import matplotlib.pyplot as plt

try:
    from nptdms import TdmsFile
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "nptdms is not installed. Install it first with: python3 -m pip install nptdms"
    ) from exc


In [ ]:
# =========================
# Configuration
# =========================
# TDMS can be a single .tdms file, a glob, or a directory containing .tdms files.
TDMS_PATHS = Path("/mnt/71/data/JINGYE/170826")

GROUP = "State"
CHANNEL = "stateu16"
RAW_FRAME_SHAPE = (1024, 1024)
BLOCK_SHAPE = (4, 4)          # 1024x1024 -> 256x256
BLOCK_REDUCER = "mean"

# Each Template_###_video.tdms contributes exactly one state image.
# 0 means first frame in each TDMS; use 4 if the fifth frame is the stable measured state.
SELECTED_FRAME_INDEX = 0

TDMS_START_INDEX = 1
TDMS_STOP_INDEX = 2964
N_SAMPLES = TDMS_STOP_INDEX - TDMS_START_INDEX + 1
N_TRAIN = 2000
WASHOUT = 200

# One-step-ahead task: state(t) predicts truth(t+1).
# Train stage runs states 1-2000. The first 200 are washout and do not train W_out.
# Ridge fit uses states 201-2000 -> truth labels 202-2001.
# Test stage: after W_out is fixed, states 2001-2964 make predictions for truth labels 2002-2965.
TRAIN_STATE_START = 1
TRAIN_STATE_STOP = N_TRAIN
TEST_STATE_START = N_TRAIN + 1
TEST_STATE_STOP = N_SAMPLES
N_TRAIN_PAIRS = TRAIN_STATE_STOP - TRAIN_STATE_START + 1
N_TEST_PAIRS = TEST_STATE_STOP - TEST_STATE_START + 1

MG_ARRAYS_PATH = Path("jy RC") / "Exp August" / "measurement_mg_arrays_t3000_tau17.npz"
RESULT_PATH = Path("streaming_tdms_rc_results_ridge_256x256.npz")

# Readout settings ported from RC_MGV1: use a fixed sampled camera/state vector,
# then solve Wout with ridge regression after the washout part of the train stage.
READOUT_FEATURE_MODE = "raw"       # "raw", "sqrt_intensity", or "binary_threshold"
N_READOUT_FEATURES = 1024          # fixed sampled features from 256x256; do not set None unless memory is enough
RANDOM_SEED = 42
RIDGE_ALPHA = 1e-4
SOLVE_JITTER = 1e-6
ADD_BIAS = False             # match RC_MGV1: y = X @ W_out, no explicit bias column
PRINT_EVERY = 100


In [ ]:
def template_index_from_path(path):
    import re
    match = re.search(r"Template_(\d+)_video\.tdms$", Path(path).name)
    if match is None:
        return None
    return int(match.group(1))


def expand_tdms_paths(tdms_paths, start_index=TDMS_START_INDEX, stop_index=TDMS_STOP_INDEX):
    if isinstance(tdms_paths, (str, Path)):
        path = Path(tdms_paths)
        text = str(path)
        if text == "PUT_YOUR_TDMS_PATH_OR_GLOB_HERE":
            raise ValueError("Please set TDMS_PATHS to your TDMS file path, directory, or glob.")
        if any(ch in text for ch in "*?[]"):
            candidates = [Path(x) for x in glob.glob(text)]
        elif path.is_dir():
            candidates = list(path.glob("Template_*_video.tdms"))
        else:
            candidates = [path]
    else:
        candidates = [Path(x) for x in tdms_paths]

    if not candidates:
        raise FileNotFoundError(f"No TDMS files matched: {tdms_paths}")

    indexed_paths = []
    plain_paths = []
    for path in candidates:
        index = template_index_from_path(path)
        if index is None:
            plain_paths.append(path)
        elif int(start_index) <= index <= int(stop_index):
            indexed_paths.append((index, path))

    if indexed_paths:
        indexed_paths.sort(key=lambda item: item[0])
        paths = [path for _, path in indexed_paths]
        found_indices = [index for index, _ in indexed_paths]
        expected_indices = set(range(int(start_index), int(stop_index) + 1))
        missing_indices = sorted(expected_indices - set(found_indices))
        if missing_indices:
            preview = ", ".join(f"{idx:03d}" for idx in missing_indices[:20])
            raise FileNotFoundError(
                f"Missing Template indices between {start_index:03d} and {stop_index:03d}: {preview}"
            )
    else:
        paths = sorted(plain_paths)

    if not paths:
        raise FileNotFoundError(
            f"No Template_###_video.tdms files found between {start_index:03d} and {stop_index:03d}."
        )

    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing TDMS file(s):\n" + "\n".join(missing[:10]))

    print(f"TDMS files to stream: {len(paths)}")
    print("First TDMS file:", paths[0])
    print("Last TDMS file:", paths[-1])
    return paths

def block_downsample_2d(frame, block_shape=(8, 8), reducer="mean"):
    frame = np.asarray(frame, dtype=np.float32)
    if frame.ndim != 2:
        raise ValueError(f"Expected 2D frame, got {frame.shape}")

    by, bx = (int(v) for v in block_shape)
    h, w = frame.shape
    if h % by or w % bx:
        raise ValueError(f"Frame shape {frame.shape} is not divisible by block_shape={block_shape}")

    blocks = frame.reshape(h // by, by, w // bx, bx)
    if reducer == "mean":
        return blocks.mean(axis=(1, 3), dtype=np.float32)
    if reducer == "sum":
        return blocks.sum(axis=(1, 3), dtype=np.float32)
    if reducer == "max":
        return blocks.max(axis=(1, 3))
    raise ValueError(f"Unknown reducer: {reducer}")


def normalize_state_image(image, eps=1e-8):
    image = np.asarray(image, dtype=np.float32)
    image = image - float(image.mean())
    std = float(image.std())
    if std > eps:
        image = image / std
    return image


def iter_tdms_raw_frames(tdms_path, group=GROUP, channel=CHANNEL, frame_shape=RAW_FRAME_SHAPE):
    frame_shape = tuple(int(v) for v in frame_shape)
    pixels_per_frame = frame_shape[0] * frame_shape[1]
    remainder = np.empty(0, dtype=np.float32)

    with TdmsFile.open(str(tdms_path)) as tdms:
        tdms_channel = tdms[group][channel]
        chunks = getattr(tdms_channel, "data_chunks", None)
        chunk_iter = chunks() if chunks is not None else [tdms_channel[:]]

        for chunk in chunk_iter:
            values = np.asarray(chunk[:], dtype=np.float32).reshape(-1)
            if remainder.size:
                values = np.concatenate([remainder, values])

            n_complete = values.size // pixels_per_frame
            usable = n_complete * pixels_per_frame
            if usable:
                frames = values[:usable].reshape(n_complete, *frame_shape)
                for frame in frames:
                    yield frame
            remainder = values[usable:]

    if remainder.size:
        raise ValueError(
            f"{tdms_path} has {remainder.size} trailing values after complete frames. "
            "Check RAW_FRAME_SHAPE/GROUP/CHANNEL."
        )


def iter_measured_state_vectors(
    tdms_paths,
    start_state=1,
    stop_state=N_SAMPLES,
    selected_frame_index=SELECTED_FRAME_INDEX,
    raw_frame_shape=RAW_FRAME_SHAPE,
    block_shape=BLOCK_SHAPE,
    block_reducer=BLOCK_REDUCER,
):
    """Stream state vectors from TDMS without storing the full state matrix.

    start_state and stop_state are 1-based inclusive state labels.
    """
    paths = expand_tdms_paths(tdms_paths)
    start_state = int(start_state)
    stop_state = int(stop_state)
    if start_state < 1 or stop_state < start_state:
        raise ValueError("Use 1-based state labels with 1 <= start_state <= stop_state.")

    state_label = 0
    for path in paths:
        for frame_i, raw_frame in enumerate(iter_tdms_raw_frames(path, frame_shape=raw_frame_shape)):
            if selected_frame_index is not None and frame_i != int(selected_frame_index):
                continue

            state_label += 1
            if state_label < start_state:
                if selected_frame_index is not None:
                    break
                continue
            if state_label > stop_state:
                return

            state_img = block_downsample_2d(raw_frame, block_shape=block_shape, reducer=block_reducer)
            state_img = normalize_state_image(state_img)
            yield state_label, state_img.reshape(-1)

            if selected_frame_index is not None:
                break

    if state_label < stop_state:
        raise ValueError(f"Only found {state_label} states, expected at least {stop_state}.")



In [ ]:
def reservoir_raw_intensity(x, sample_idx=None):
    """Scheme 1 from RC_MGV1: r = I, adapted to measured state vectors."""
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if sample_idx is None:
        return x
    return x[np.asarray(sample_idx, dtype=int)]


def reservoir_amplitude(x, sample_idx=None):
    """Scheme 2 from RC_MGV1: r = sqrt(I). Clips measured intensities at zero."""
    x = reservoir_raw_intensity(x, sample_idx)
    return np.sqrt(np.clip(x, 0.0, None)).astype(np.float32)


def reservoir_binary_threshold(x, sample_idx=None, threshold=None):
    """Scheme 3 from RC_MGV1: r = H(I - A)."""
    x = reservoir_raw_intensity(x, sample_idx)
    if threshold is None:
        threshold = float(np.median(x))
    return (x > threshold).astype(np.float32), threshold


def make_sample_idx(n_total_features, n_readout_features=N_READOUT_FEATURES, seed=RANDOM_SEED):
    n_total_features = int(n_total_features)
    if n_readout_features is None or int(n_readout_features) >= n_total_features:
        return None
    rng = np.random.default_rng(int(seed))
    return np.sort(rng.choice(n_total_features, size=int(n_readout_features), replace=False))


def transform_readout_features(x, sample_idx=None, mode=READOUT_FEATURE_MODE, threshold=None):
    if mode == "raw":
        return reservoir_raw_intensity(x, sample_idx), threshold
    if mode == "sqrt_intensity":
        return reservoir_amplitude(x, sample_idx), threshold
    if mode == "binary_threshold":
        return reservoir_binary_threshold(x, sample_idx, threshold=threshold)
    raise ValueError(f"Unknown READOUT_FEATURE_MODE: {mode}")


In [ ]:
class StreamingRidgeMeasuredStateRC:
    """Train Wout from streamed measured reservoir states using ridge regression.

    This mirrors the successful RC_MGV1 readout pattern more closely than the
    previous one-pass NLMS update: fixed readout features, optional washout,
    optional bias, and a closed-form ridge solve. Only X.T @ X and X.T @ y are
    accumulated, so the full TDMS state matrix is never stored.
    """

    def __init__(
        self,
        n_features,
        ridge_alpha=1e-4,
        solve_jitter=1e-6,
        add_bias=False,
        feature_mode="raw",
        sample_idx=None,
        threshold=None,
    ):
        self.n_features = int(n_features)
        self.ridge_alpha = float(ridge_alpha)
        self.solve_jitter = float(solve_jitter)
        self.add_bias = bool(add_bias)
        self.feature_mode = str(feature_mode)
        self.sample_idx = sample_idx
        self.threshold = threshold
        self.W_out = None
        self.bias = 0.0

    @property
    def n_model_features(self):
        return self.n_features + int(self.add_bias)

    def _features(self, x, update_threshold=False):
        z, threshold = transform_readout_features(
            x,
            sample_idx=self.sample_idx,
            mode=self.feature_mode,
            threshold=self.threshold,
        )
        if update_threshold and self.feature_mode == "binary_threshold" and self.threshold is None:
            self.threshold = threshold
        z = np.asarray(z, dtype=np.float64).reshape(-1)
        if z.shape[0] != self.n_features:
            raise ValueError(f"Expected {self.n_features} readout features, got {z.shape[0]}")
        if self.add_bias:
            z = np.concatenate([z, np.ones(1, dtype=np.float64)])
        return z

    def fit_stream(self, state_stream_factory, truth, washout=0, print_every=100):
        truth = np.asarray(truth, dtype=np.float64)
        n = self.n_model_features
        xtx = np.zeros((n, n), dtype=np.float64)
        xty = np.zeros(n, dtype=np.float64)
        train_pred_before_fit = []
        train_truth = []
        used_labels = []

        for step_i, (state_label, x) in enumerate(state_stream_factory(), start=1):
            if step_i <= int(washout):
                continue

            target_label = state_label + 1
            y_true = float(truth[target_label - 1])
            z = self._features(x, update_threshold=True)

            xtx += np.outer(z, z)
            xty += z * y_true
            train_truth.append(y_true)
            used_labels.append(state_label)

            if print_every and len(train_truth) % int(print_every) == 0:
                print(f"accumulated {len(train_truth)} ridge samples through state {state_label}")

        if not train_truth:
            raise ValueError("No training samples remained after washout. Reduce WASHOUT or check train range.")

        penalty = self.ridge_alpha * np.eye(n, dtype=np.float64)
        jitter = self.solve_jitter * np.eye(n, dtype=np.float64)
        if self.add_bias:
            penalty[-1, -1] = 0.0
            jitter[-1, -1] = 0.0
        w = np.linalg.solve(xtx + penalty + jitter, xty)

        if self.add_bias:
            self.W_out = w[:-1]
            self.bias = float(w[-1])
        else:
            self.W_out = w
            self.bias = 0.0

        used_label_set = set(used_labels)
        for state_label, x in state_stream_factory():
            if state_label not in used_label_set:
                continue
            train_pred_before_fit.append(self.predict_one(x))

        return (
            np.asarray(used_labels, dtype=np.int64),
            np.asarray(train_pred_before_fit, dtype=np.float64),
            np.asarray(train_truth, dtype=np.float64),
        )

    def predict_one(self, x):
        if self.W_out is None:
            raise RuntimeError("Call fit_stream() before prediction.")
        z, _ = transform_readout_features(
            x,
            sample_idx=self.sample_idx,
            mode=self.feature_mode,
            threshold=self.threshold,
        )
        z = np.asarray(z, dtype=np.float64).reshape(-1)
        return float(z @ self.W_out + self.bias)

    def predict_stream(self, state_stream_factory):
        labels = []
        preds = []
        for state_label, x in state_stream_factory():
            labels.append(state_label)
            preds.append(self.predict_one(x))
        return np.asarray(labels, dtype=np.int64), np.asarray(preds, dtype=np.float64)


def regression_metrics(y_pred, y_true):
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    err = y_pred - y_true
    rmse = np.sqrt(np.mean(err ** 2))
    corr = np.nan if y_pred.size < 2 else np.corrcoef(y_pred, y_true)[0, 1]
    return {
        "mse": float(np.mean(err ** 2)),
        "rmse": float(rmse),
        "nrmse": float(rmse / (np.std(y_true) + 1e-12)),
        "mae": float(np.mean(np.abs(err))),
        "corr": float(corr),
    }


In [ ]:
# =========================
# Load truth values only
# =========================
print("Current working directory:", Path.cwd())
print("MG arrays path:", MG_ARRAYS_PATH)
if not MG_ARRAYS_PATH.exists():
    raise FileNotFoundError(f"MG_ARRAYS_PATH does not exist: {MG_ARRAYS_PATH.resolve()}")
mg = np.load(MG_ARRAYS_PATH)
truth_values = np.asarray(mg["truth_values"], dtype=np.float64)

needed_truth_stop = TEST_STATE_STOP + 1
if truth_values.shape[0] < needed_truth_stop:
    raise ValueError(f"Need truth_values through label {needed_truth_stop}, got {truth_values.shape[0]}")

DOWNSAMPLED_SHAPE = (RAW_FRAME_SHAPE[0] // BLOCK_SHAPE[0], RAW_FRAME_SHAPE[1] // BLOCK_SHAPE[1])
N_DOWNSAMPLED_FEATURES = DOWNSAMPLED_SHAPE[0] * DOWNSAMPLED_SHAPE[1]
SAMPLE_IDX = make_sample_idx(N_DOWNSAMPLED_FEATURES, N_READOUT_FEATURES, seed=RANDOM_SEED)
N_FEATURES = N_DOWNSAMPLED_FEATURES if SAMPLE_IDX is None else len(SAMPLE_IDX)

print("TDMS file index range:", f"Template_{TDMS_START_INDEX:03d}_video.tdms", "to", f"Template_{TDMS_STOP_INDEX:03d}_video.tdms")
print("number of TDMS states:", N_SAMPLES)
print("downsampled state image shape:", DOWNSAMPLED_SHAPE)
print("downsampled feature count:", N_DOWNSAMPLED_FEATURES)
print("readout feature mode:", READOUT_FEATURE_MODE)
print("sampled readout feature count:", N_FEATURES)
print("washout states skipped before fitting W_out:", WASHOUT)
print("one-step task: state(t) -> truth(t+1)")
print("train state labels:", TRAIN_STATE_START, "to", TRAIN_STATE_STOP)
print("train truth labels:", TRAIN_STATE_START + 1, "to", TRAIN_STATE_STOP + 1)
print("test state labels:", TEST_STATE_START, "to", TEST_STATE_STOP)
print("test truth labels:", TEST_STATE_START + 1, "to", TEST_STATE_STOP + 1)


In [ ]:
# =========================
# Train stage: stream states 1-2000, discard first 200 as washout, then solve W_out
# =========================
def train_state_stream():
    return iter_measured_state_vectors(
        TDMS_PATHS,
        start_state=TRAIN_STATE_START,
        stop_state=TRAIN_STATE_STOP,
        selected_frame_index=SELECTED_FRAME_INDEX,
        raw_frame_shape=RAW_FRAME_SHAPE,
        block_shape=BLOCK_SHAPE,
        block_reducer=BLOCK_REDUCER,
    )

rc = StreamingRidgeMeasuredStateRC(
    n_features=N_FEATURES,
    ridge_alpha=RIDGE_ALPHA,
    solve_jitter=SOLVE_JITTER,
    add_bias=ADD_BIAS,
    feature_mode=READOUT_FEATURE_MODE,
    sample_idx=SAMPLE_IDX,
)

train_state_labels_fit, y_out, y_train_seen = rc.fit_stream(
    train_state_stream,
    truth_values,
    washout=WASHOUT,
    print_every=PRINT_EVERY,
)
# Same meaning as RC_MGV1: after W_out is solved, y_out = X_fit @ W_out for states after washout.
train_pred = y_out
train_truth_labels_fit = train_state_labels_fit + 1

train_metrics = regression_metrics(train_pred, y_train_seen)
print("Train metrics after ridge solve:")
for key, value in train_metrics.items():
    print(f"  {key}: {value:.6g}")


In [ ]:
# =========================
# Test stage: keep W_out fixed and predict from later measured states
# =========================
def test_state_stream():
    return iter_measured_state_vectors(
        TDMS_PATHS,
        start_state=TEST_STATE_START,
        stop_state=TEST_STATE_STOP,
        selected_frame_index=SELECTED_FRAME_INDEX,
        raw_frame_shape=RAW_FRAME_SHAPE,
        block_shape=BLOCK_SHAPE,
        block_reducer=BLOCK_REDUCER,
    )

test_labels, test_pred = rc.predict_stream(test_state_stream)
y_test_labels = test_labels + 1
y_test = truth_values[y_test_labels - 1]

test_metrics = regression_metrics(test_pred, y_test)
print("Test metrics:")
for key, value in test_metrics.items():
    print(f"  {key}: {value:.6g}")

np.savez(
    RESULT_PATH,
    W_out=rc.W_out,
    bias=rc.bias,
    sample_idx=np.array([], dtype=np.int64) if SAMPLE_IDX is None else SAMPLE_IDX,
    train_state_labels=train_state_labels_fit,
    train_truth_labels=train_truth_labels_fit,
    train_pred=train_pred,
    y_train=y_train_seen,
    test_state_labels=test_labels,
    test_truth_labels=y_test_labels,
    test_pred=test_pred,
    y_test=y_test,
    train_metrics=np.array(train_metrics, dtype=object),
    test_metrics=np.array(test_metrics, dtype=object),
    ridge_alpha=RIDGE_ALPHA,
    solve_jitter=SOLVE_JITTER,
    washout=WASHOUT,
    readout_feature_mode=READOUT_FEATURE_MODE,
    n_readout_features=N_FEATURES,
    raw_frame_shape=np.array(RAW_FRAME_SHAPE),
    block_shape=np.array(BLOCK_SHAPE),
)
print("Saved readout and prediction result:", RESULT_PATH)


In [ ]:
# =========================
# Plot truth vs train/test prediction
# =========================
truth_plot_labels = np.arange(1, TEST_STATE_STOP + 2)
truth_plot = truth_values[:TEST_STATE_STOP + 1]

fig, axes = plt.subplots(4, 1, figsize=(14, 12), constrained_layout=True)

axes[0].axis("off")
summary = (
    f"Ridge measured-state readout\n"
    f"mode={READOUT_FEATURE_MODE}, features={N_FEATURES}, washout={WASHOUT}, ridge={RIDGE_ALPHA:g}\n"
    f"train RMSE={train_metrics['rmse']:.4g}, train NRMSE={train_metrics['nrmse']:.4g}, train corr={train_metrics['corr']:.4g}\n"
    f"test RMSE={test_metrics['rmse']:.4g}, test NRMSE={test_metrics['nrmse']:.4g}, test corr={test_metrics['corr']:.4g}"
)
axes[0].text(0.01, 0.75, summary, va="top", ha="left", fontsize=11, family="monospace")

axes[1].plot(truth_plot_labels, truth_plot, color="black", lw=0.9, label="truth value")
axes[1].plot(train_truth_labels_fit, train_pred, color="tab:blue", lw=0.8, label="train prediction after washout")
axes[1].plot(y_test_labels, test_pred, color="tab:red", lw=0.9, label="test prediction")
axes[1].axvline(N_TRAIN + 1.5, color="0.45", ls="--", lw=1)
axes[1].set_title("Measured-state RC: one-step state(t) -> truth(t+1)")
axes[1].set_xlabel("truth label")
axes[1].set_ylabel("normalized truth value")
axes[1].legend()
axes[1].grid(alpha=0.25)

axes[2].plot(train_truth_labels_fit, y_train_seen, color="black", lw=0.9, label="truth value")
axes[2].plot(train_truth_labels_fit, train_pred, color="tab:blue", lw=0.8, label="train prediction")
axes[2].set_title(f"Training segment after washout, RMSE={train_metrics['rmse']:.4g}, NRMSE={train_metrics['nrmse']:.4g}")
axes[2].set_xlabel("truth label")
axes[2].set_ylabel("normalized truth value")
axes[2].legend()
axes[2].grid(alpha=0.25)

axes[3].plot(y_test_labels, y_test, color="black", lw=1.0, label="truth value")
axes[3].plot(y_test_labels, test_pred, color="tab:red", lw=1.0, label="test prediction")
axes[3].set_title(f"Test segment, RMSE={test_metrics['rmse']:.4g}, NRMSE={test_metrics['nrmse']:.4g}")
axes[3].set_xlabel("truth label")
axes[3].set_ylabel("normalized truth value")
axes[3].legend()
axes[3].grid(alpha=0.25)

plt.show()
